# Lesson 05: The Tool Use Design Pattern

In this notebook, you will:
1. Implement `get_current_time` with `ZoneInfo` and Gemini function calling (`google-genai`).
2. Build a Contoso SQLite sales analytics tool with a strict read-only SQL guard.
3. Test tool-calling smoke assertions.

In [ ]:
!pip install -q -U google-genai mcp python-dotenv

In [ ]:
import os, json, re, sqlite3
from datetime import datetime
from zoneinfo import ZoneInfo
from google import genai
from google.genai import types

TIMEZONES = {"san francisco": "America/Los_Angeles", "tokyo": "Asia/Tokyo", "london": "Europe/London"}

def get_current_time(location: str) -> str:
    """Get the current time for a given city."""
    tz = TIMEZONES.get(location.lower())
    if not tz:
        return json.dumps({"location": location, "current_time": "unknown"})
    return json.dumps({"location": location, "current_time": datetime.now(ZoneInfo(tz)).strftime("%I:%M %p")})

print(get_current_time("San Francisco"))

In [ ]:
# Read-only SQLite Contoso Sales Tool
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE sales (region TEXT, product TEXT, revenue REAL)")
conn.executemany("INSERT INTO sales VALUES (?, ?, ?)", [
    ("North", "Cloud Gateway", 12400.0),
    ("Europe", "Edge Sensor", 9800.0),
])

def fetch_sales_data_using_sqlite_query(sql: str) -> str:
    """Execute a read-only SELECT query on the Contoso sales database."""
    if not re.match(r"^\s*SELECT\b", sql, re.IGNORECASE) or re.search(r"\b(DROP|DELETE|UPDATE|INSERT|ALTER)\b", sql, re.IGNORECASE):
        return json.dumps({"error": "Blocked non-SELECT query"})
    return json.dumps(conn.execute(sql).fetchall())

print(fetch_sales_data_using_sqlite_query("SELECT region, revenue FROM sales"))